In [1]:
import scanpy as sc
import hdf5plugin
import anndata as ad

import numpy as np
from numpy.random import choice
import pandas as pd

import matplotlib.pyplot as plt
import seaborn.objects  as so

from scipy.stats import chi2_contingency, pearsonr
import decoupler as dc

from tqdm import tqdm

In [163]:
import importlib
importlib.reload(dc)

<module 'decoupler' from '/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/decoupler/__init__.py'>

# Load Data:

In [2]:
def ctrl_pert_split_dataset(norm_data: ad.AnnData, bad_perts : list[str] = None) -> tuple[list[str], dict[str,list[str]]]:
    """
    Split the norm_data.obs dataframe into control cells and each perturbation.

    Parameters
    ----------
    norm_data : anndata
        The anndata object containing the data.
    bad_perts : list
        A list of perturbations at least one of the models cannot process.

    Returns
    -------
    ctrl_i_vec : list[str]
        A vector of all indices of control cells in norm_data.obs (corresponds to the rows in norm_data.X).
    pert_i_dict : dict[str, list[str]]
        A dictionary with the perturbation name and the indices in norm_data.obs where this perturbation is found as key value pairs.
    """
    # Split df into perturbed and control cells
    
    by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]
    # Find indices for each perturbation
    pert_i_dict = {pert.iloc[0]["perturbation"]:pert.index.to_list() for pert in by_pert}
    # Extract the control cells separately
    ctrl_i_vec = pert_i_dict.pop("ctrl")

    # Optional: remove all perturbations that do not appear in a dataset
    if bad_perts is not None:
        for bp in bad_perts:
            pert_i_dict.pop(bp[:-5]) # the last 5 letters are +ctrl and need to be ignored
    
    return (ctrl_i_vec, pert_i_dict)

In [3]:
names = ["Norman19"]#,"Replogle22"]
models = ["pGRiNS", "Random"]
tests = ["Common DEGs", "Common highly expressed genes"]

In [ ]:
"""
pgrins_full = sc.read_h5ad("../Data/Projects/KeggoRo/perturb_norm_pert_reduced.h5ad")
pgrins = {}
pgrins["Norman19"] = pgrins_full[pgrins_full.obs["PertNum"]<=75]
pgrins["Replogle22"] = pgrins_full[pgrins_full.obs["PertNum"]==-1 | pgrins_full.obs["PertNum"]>75]
"""
exp = {}
exp["Norman19"] = sc.read_h5ad("../Data/Experimental/Norman19/perturb_norm_subset_KeggoRo.h5ad")
exp["Replogle22"] = sc.read_h5ad("../Data/Experimental/Replogle22/perturb_norm_subset_KeggoRo.h5ad")

perts = {}
pert_indices = {}
pert_means = {}
for name in names:
    perts[name] = list(exp[name].obs["perturbation"].unique())
    perts[name].remove("ctrl")
    pert_indices[name] = ctrl_pert_split_dataset(exp[name])
    pert_means[name] = {pert : np.asarray(np.mean(exp[name][pert_indices[name][1][pert]].layers["log1p"],axis=0)).squeeze() for pert in perts[name]}
    #sc.tl.rank_genes_groups(exp[name],groupby="perturbation",reference="ctrl",rankby_abs=True,layer="log1p")
    #sc.tl.rank_genes_groups(pgrins[name],groupby="perturbation",reference="ctrl",layer="log1p")

/tmp/ipykernel_235977/733474065.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]


# Visual comparison:

In [ ]:
# visualization method: heatmap

# Shared DEGs:

## Get genes:

In [5]:
# Returns a dict of dicts: each dataset has a dict with perturbations as keys and the DEGs for these perturbations as values
def get_degs(adata_dict):
    degs = {}
    for name in names:
        degs[name] = {}
        for pert in tqdm(perts[name]):
            degs[name][pert] = adata_dict[name].uns["rank_genes_groups"]["names"][pert][adata_dict[name].uns["rank_genes_groups"]["pvals_adj"][pert] < 0.05]
    return degs

In [ ]:
# Get genes whose mean expression value in a certain perturbation is higher than the mean of means
def get_on_genes(adata_dict):
    on_genes = {}
    for name in names:
        on_genes[name] = {}
        for pert in tqdm(perts[name]):
            on_genes[name][pert] = adata_dict[name].var_names[pert_means[name][pert]>np.mean(pert_means[name][pert])]
    return on_genes

In [7]:
# For negative control: get random number of genes equal to no. of genes per pert in pGRiNS
def get_rand_dict(gene_dict):
    dict_rand = {}
    for name in names:
        dict_rand[name] = {}
        for pert in tqdm(perts[name]):
            dict_rand[name][pert] = choice(list(pgrins[name].var_names),size=len(gene_dict[name][pert]))
    return dict_rand

In [8]:
exp_dict = {} # Has structure test -> name -> pert
model_dict = {} # Has structure test -> model -> name -> pert

exp_dict[tests[0]] = get_degs(exp)
#exp_dict[tests[1]] = get_on_genes(exp)
"""
model_dict[tests[0]] = {}
model_dict[tests[1]] = {}
model_dict[tests[0]][models[0]] = get_degs(pgrins)
model_dict[tests[1]][models[0]] = get_on_genes(pgrins)
model_dict[tests[0]][models[1]] = get_rand_dict(model_dict[tests[0]][models[0]])
model_dict[tests[1]][models[1]] = get_rand_dict(model_dict[tests[1]][models[0]])
"""

100%|██████████| 963/963 [00:00<00:00, 9249.13it/s]


'\nmodel_dict[tests[0]] = {}\nmodel_dict[tests[1]] = {}\nmodel_dict[tests[0]][models[0]] = get_degs(pgrins)\nmodel_dict[tests[1]][models[0]] = get_on_genes(pgrins)\nmodel_dict[tests[0]][models[1]] = get_rand_dict(model_dict[tests[0]][models[0]])\nmodel_dict[tests[1]][models[1]] = get_rand_dict(model_dict[tests[1]][models[0]])\n'

## Chi squared:

In [ ]:
chi2_res_all = {} # Has structure test -> name -> model (list of pert_score)
for test in tests:
    chi2_res_all[test] = {}
    for name in names:
        chi2_res_all[test][name] = {}
        for model in models:
            chi2_res_all[test][name][model] = []
            for pert in perts[name]:
                degs_exp_set = set(exp_dict[test][name][pert])
                degs_model_set = set(model_dict[model][name][pert])
                all_genes = set(exp[name].var_names)

                cont_matrix = np.array([[len(degs_exp_set & degs_pgrins_set),len(degs_pgrins_set - degs_exp_set)],[len(degs_exp_set - degs_pgrins_set),len(all_genes - (degs_exp_set|degs_pgrins_set))]])
                res = chi2_contingency(cont_matrix)
                chi2_res_all[test][name][model].append(res.pvalue)

In [ ]:
# plot chi squared pval hist for perts
# for comparison: randomly assign genes as DEGs and calculate chi squared stat between that and exp data
# then do t test (or wilcoxon or sth) between distributions

In [ ]:
# How many genes are ON/OFF in both?

# Compare using metrics:

## Interpolated mean:

In [ ]:
# plot hist of distance between interpolated µ and each pert mean

In [ ]:
# or: across perts: plot mean MSE between adata_mean and adata_cell, and adata_mean and pgrins_cell

## Weighted metrics:

In [ ]:
# use WMSE, WR2, etc. to compare µ_c,exp and each µ_p,syn to GT of pert

# For weights calculation of interpolated duplicate etc. in GRiNS: is the order of pval_adj or t score alphabetically? Or are they reordered according to pval?

## GSEA:

- Inspired from https://www.sc-best-practices.org/conditions/gsea_pathway.html

In [5]:
# Retrieving via python
msigdb = dc.op.resource("MSigDB")

# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))].rename(columns={"genesymbol":"target","geneset":"source"})
"""
# Retrieving via python
msigdb = dc.get_resource("MSigDB")

# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]#.rename(columns={"genesymbol":"target","geneset":"source"})
"""

'\n# Retrieving via python\nmsigdb = dc.get_resource("MSigDB")\n\n# Get reactome pathways\nreactome = msigdb.query("collection == \'reactome_pathways\'")\n# Filter duplicates\nreactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]#.rename(columns={"genesymbol":"target","geneset":"source"})\n'

In [6]:
gsea_dict = {}
for name in names:
    gsea_dict[name] = {}
    # Filter reactome by genes in each dataset
    gsea_dict[name]["reactome"] = reactome[reactome["target"].isin(exp[name].var_names)]
    # Get the genesets with no. of genes in [15,500]
    geneset_size = gsea_dict[name]["reactome"].groupby("source").size()
    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets

    for pert in tqdm(perts[name][:10]):
        t_stats = sc.get.rank_genes_groups_df(exp[name], pert).set_index("names").sort_values("scores", key=np.abs, ascending=False)[["scores"]].rename_axis([pert], axis=1)
        scores, pvals = dc.mt.gsea(
            t_stats.T,
            gsea_dict[name]["reactome"][gsea_dict[name]["reactome"]["source"].isin(gsea_dict[name]["genesets"])],
        )
        gsea_dict[name][pert] = (
            pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
            .droplevel(level=1, axis=1)
            .sort_values("pval")
        )
        
"""
gsea_dict = {}
for name in names:
    gsea_dict[name] = {}
    # Filter reactome by genes in each dataset
    gsea_dict[name]["reactome"] = reactome[reactome["genesymbol"].isin(exp[name].var_names)]
    # Get the genesets with no. of genes in [15,500]
    geneset_size = gsea_dict[name]["reactome"].groupby("geneset").size()
    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets
"""

100%|██████████| 10/10 [01:30<00:00,  9.07s/it]


'\ngsea_dict = {}\nfor name in names:\n    gsea_dict[name] = {}\n    # Filter reactome by genes in each dataset\n    gsea_dict[name]["reactome"] = reactome[reactome["genesymbol"].isin(exp[name].var_names)]\n    # Get the genesets with no. of genes in [15,500]\n    geneset_size = gsea_dict[name]["reactome"].groupby("geneset").size()\n    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets\n'

In [ ]:
# get rank of pathways
# plot median distance between ranks for GRiNS and ranks for pert

## Coexpression Graph:
- Inspired from https://link.springer.com/protocol/10.1007/978-1-0716-2067-0_19
- https://cox-labs.github.io/coxdocs/WGCNA.html

In [ ]:
# Idea: use it to find modules and check them for pathways
# Other idea: look at Genetik Übung again and follow that (Codina)
# Another idea: confusion matrix containing overlap of genes per module between exp and pgrins

In [ ]:
for name in names:
    pert_mean_matrix = np.vstack(list(pert_means[name].values()))
    pd.DataFrame(pert_mean_matrix,columns=exp[name].var_names).to_csv(f"Data/Experimental/{name}/pert_mean.csv",sep=" ",index=False)
cor = pd.DataFrame(pert_mean_matrix,columns=exp[name].var_names).corr()

In [22]:
%%R -i cor
library("WGCNA")
powers = c(c(1:10), seq(from = 12, to=30, by=2))
sft = pickSoftTreshold(cor, powerVector=powers,verbose=5)

plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2], xlab="Soft Threshold power)", ylab="Scale Free Topology Model Fit,signed R^2", type="n", main = paste("Scale independence"));

text(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2], labels=powers,cex=cex1,col="red");

abline(h=0.80, col="red")

plot(sft$fitIndices[,1], sft$fitIndices[,5], xlab="Soft Threshold (power)", ylab="Mean Connectivity", type="n", main = paste("Mean connectivity"))

text(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=cex1,col="red")

Error in pickSoftTreshold(cor, powerVector = powers, verbose = 5) : 
  could not find function "pickSoftTreshold"


RInterpreterError: Failed to parse and evaluate line 'library("WGCNA")\npowers = c(c(1:10), seq(from = 12, to=30, by=2))\nsft = pickSoftTreshold(cor, powerVector=powers,verbose=5)\n\nplot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2], xlab="Soft Threshold power)", ylab="Scale Free Topology Model Fit,signed R^2", type="n", main = paste("Scale independence"));\n\ntext(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2], labels=powers,cex=cex1,col="red");\n\nabline(h=0.80, col="red")\n\nplot(sft$fitIndices[,1], sft$fitIndices[,5], xlab="Soft Threshold (power)", ylab="Mean Connectivity", type="n", main = paste("Mean connectivity"))\n\ntext(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=cex1,col="red")\n'.
R error message: 'Error in pickSoftTreshold(cor, powerVector = powers, verbose = 5) : \n  could not find function "pickSoftTreshold"'

In [ ]:
# Compare networks for exp & pGRiNS with CoDiNA: file:///home/gesomme/Documents/Uni/B_Semester_5/Genetik/%C3%9Cbung/Tutorial%20on%20Gene%20Expression%20and%20Network%20Analysis.html
# Look at modules and perform GO enrichment analysis or sth

# GSEA is done perturbation wise (can pGRiNS capture the specific pathways affected by a perturbation?)
# GO is done across perturbations (are genes generally coexpressed in such a way that networks/pathways are derivable?   )

## GO Enrichment: